# Malta Dive Explorer

To explore shipwrecks while diving in Malta, I wanted to find out what dive sites are available, how deep they are, and which ones are rated the highest.

**Source:** [maltadives.com](https://maltadives.com/sites/en)  
**Tools:** Python, Pandas, Folium, Plotly, geopy

## 1. Load Data
Data was scraped directly from maltadives.com and contains 233 dive sites across Malta, Gozo and Comino.

In [1]:
import pandas as pd
import plotly.express as px
import folium
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

In [2]:
tables = pd.read_html("https://maltadives.com/sites/en")
df = tables[0]
df.head()

,#,Island,Location,Site,Interest,Popularity,# of Ratings,Avg,Shore Access,Qualification,Depth Avg,Depth Max,Description
0,1,Malta,Cirkewwa,NaN,Cave & Reef & Wall & Wreck,High,5,4.2,Easy,Beginner,5+m,30+m,Cirkewwa has many popular dive sites in Malta:...
1,2,Malta,Cirkewwa,"P29 (Patrol Boat P29, Boltenhagen)",Wreck,High,10,4.4,Easy,Advanced,20m,34m,Patrol Boat P29 (Boltenhagen) is a popular wre...
2,3,Malta,Cirkewwa,"Rozi (MV Rozi, Tug Boat Rozi, Rozy)",Reef & Wreck,High,7,4.4,Easy,Advanced,20m,34m,Tugboat Rozi is a popular wreck dive in Malta....
3,4,Malta,Cirkewwa,"Arch (Cirkewwa Arch, Right Arch, Green Arch Re...",Cave & Reef,High,7,3.3,Easy,Beginner,15m,25m,Cirkewwa Arch dive site is a beautiful natural...
4,5,Malta,Cirkewwa,Paradise Bay (Left Reef),Reef,Medium,4,3.3,Easy,Beginner,10m,30m,Paradise Bay reef dive in Cirkewwa offers rock...


## 2. Explore Data

In [3]:
df.shape

(233, 13)

In [4]:
df.sort_values("Avg", ascending=False).head(10)

,#,Island,Location,Site,Interest,Popularity,# of Ratings,Avg,Shore Access,Qualification,Depth Avg,Depth Max,Description
13,14,Malta,Wied iz-Zurrieq (Blue Grotto),NaN,Cave & Reef & Wall & Wreck,High,1,5.0,Easy,Beginner,5+m,30+m,Wied iz-Zurrieq is famous for Blue Grotto & Um...
203,204,Gozo,NaN,Ta'Camma Caves (Ta’Cama Caves),Cave & Wall,Low,1,5.0,Boat only,Advanced,25+m,35+m,NaN
143,144,Gozo,Dwejra,NaN,Cave & Reef & Wall,High,3,5.0,Medium,Advanced,15+m,40+m,Dwejra is the most popular scuba diving site i...
122,123,Malta,NaN,"HMS Olympus (N35, Olympus)",Wreck,Low,3,5.0,Boat only,Technical,115m,130m,HMS Olympus is WWII submarine wreck in Malta. ...
112,113,Malta,NaN,HMS Nasturtium (Nasturtium),Wreck,Low,3,5.0,Boat only,Technical,65m,68m,HMS Nasturtium is WWI wreck dive in Malta. Nas...
185,186,Gozo,NaN,Ghar id-Dar,Cave,Low,1,5.0,Boat only,Beginner,NaN,NaN,Ghar id-dar is next to Wied Ir-Raheb in Gozo. ...
202,203,Gozo,NaN,Wied ir-Raheb,Cave,Low,1,5.0,Boat only,Advanced,25m,70+m,NaN
139,140,Malta,NaN,"HMS Urge (Urge, N17)",Wreck,Not dived,2,5.0,Boat only,Technical,130m,130m,HMS Urge is WWII submarine wreck in Malta. Urg...
211,212,Gozo,NaN,Phoenician Shipwreck (Tower Wreck),Wreck,Not dived,1,5.0,Boat only,Technical,105m,110m,Phoenician shipwreck is 2700 years old wreck n...
59,60,Malta,NaN,Harq Hamiem Cave (Ghar Harq il-Hammiem),Cave,Not dived,2,5.0,Hard,Technical,20m,52m,Harq Hammiem Cave in St. Julian's is the only ...


## 3. Data Cleaning
The "Depth Max" column contained text values like "34m" or "30+m". These were converted to numeric values to enable statistical analysis.

In [5]:
df["Depth Max Clean"] = (
    df["Depth Max"]
    .str.extract(r"(\d+)")[0]
    .astype(float)
)

df.head()

,#,Island,Location,Site,Interest,Popularity,# of Ratings,Avg,Shore Access,Qualification,Depth Avg,Depth Max,Description,Depth Max Clean
0,1,Malta,Cirkewwa,NaN,Cave & Reef & Wall & Wreck,High,5,4.2,Easy,Beginner,5+m,30+m,Cirkewwa has many popular dive sites in Malta:...,30.0
1,2,Malta,Cirkewwa,"P29 (Patrol Boat P29, Boltenhagen)",Wreck,High,10,4.4,Easy,Advanced,20m,34m,Patrol Boat P29 (Boltenhagen) is a popular wre...,34.0
2,3,Malta,Cirkewwa,"Rozi (MV Rozi, Tug Boat Rozi, Rozy)",Reef & Wreck,High,7,4.4,Easy,Advanced,20m,34m,Tugboat Rozi is a popular wreck dive in Malta....,34.0
3,4,Malta,Cirkewwa,"Arch (Cirkewwa Arch, Right Arch, Green Arch Re...",Cave & Reef,High,7,3.3,Easy,Beginner,15m,25m,Cirkewwa Arch dive site is a beautiful natural...,25.0
4,5,Malta,Cirkewwa,Paradise Bay (Left Reef),Reef,Medium,4,3.3,Easy,Beginner,10m,30m,Paradise Bay reef dive in Cirkewwa offers rock...,30.0


In [6]:
df.columns = (
    df.columns
    .str.lower()
    .str.replace(" ", "_")
)

df.head()

,#,island,location,site,interest,popularity,#_of_ratings,avg,shore_access,qualification,depth_avg,depth_max,description,depth_max_clean
0,1,Malta,Cirkewwa,NaN,Cave & Reef & Wall & Wreck,High,5,4.2,Easy,Beginner,5+m,30+m,Cirkewwa has many popular dive sites in Malta:...,30.0
1,2,Malta,Cirkewwa,"P29 (Patrol Boat P29, Boltenhagen)",Wreck,High,10,4.4,Easy,Advanced,20m,34m,Patrol Boat P29 (Boltenhagen) is a popular wre...,34.0
2,3,Malta,Cirkewwa,"Rozi (MV Rozi, Tug Boat Rozi, Rozy)",Reef & Wreck,High,7,4.4,Easy,Advanced,20m,34m,Tugboat Rozi is a popular wreck dive in Malta....,34.0
3,4,Malta,Cirkewwa,"Arch (Cirkewwa Arch, Right Arch, Green Arch Re...",Cave & Reef,High,7,3.3,Easy,Beginner,15m,25m,Cirkewwa Arch dive site is a beautiful natural...,25.0
4,5,Malta,Cirkewwa,Paradise Bay (Left Reef),Reef,Medium,4,3.3,Easy,Beginner,10m,30m,Paradise Bay reef dive in Cirkewwa offers rock...,30.0


In [7]:
df.isnull().sum()

#                    0
island               0
location           124
site                21
interest            15
popularity           0
#_of_ratings         0
avg                 32
shore_access         0
qualification        0
depth_avg           10
depth_max            6
description         88
depth_max_clean      6
dtype: int64

In [8]:
df.duplicated().sum()

0

## 4. Add Geographic Data
GPS coordinates were added via geocoding with geopy and saved to a CSV file. 
The pre-geocoded dataset is loaded here to avoid API rate limits.

In [10]:
df_map = pd.read_csv("malta_dive_sites_with_locations.csv")
df_map.head()

FileNotFoundError: [Errno 2] No such file or directory: 'malta_dive_sites_with_locations.csv'

In [ ]:
df_map.to_csv(
    "malta_dive_sites_with_locations.csv",
    index=False
)

## 5. Interactive Map
The map shows all dive sites with available coordinates. Color and markers indicate difficulty level.

In [ ]:
map_test = folium.Map(
    location=[35.9, 14.5],
    zoom_start=11
)

In [ ]:
df_with_coords = df_map.dropna(
    subset=["latitude", "longitude"]
)

len(df_with_coords)

93

In [ ]:
for index, row in df_with_coords.iterrows():
    folium.Marker(
        location=[
            row["latitude"],
            row["longitude"]
        ],
        popup=row["site"]
    ).add_to(map_test)

map_test

## 6. Analysis
Relationship between maximum depth, rating and difficulty — deeper sites tend to be reserved for Advanced or Technical divers.

In [ ]:
import plotly.express as px

fig = px.scatter(
    df_with_coords,
    x="depth_max_clean",
    y="avg",
    color="qualification",
    hover_name="site",
    labels={
        "depth_max_clean": "Maximum Depth (m)",
        "avg": "Average Rating",
        "qualification": "Qualification"
    },
    title="Dive Site Ratings by Depth and Qualification"
)

fig.show()

## 7. Conclusion & Next Steps

**Key findings:**
- Gozo has on average deeper and more challenging dive sites than Malta
- Wrecks tend to be rated higher than reefs
- Many top-rated sites are only accessible by boat

**Next steps:**
- Add missing GPS coordinates manually
- Add filter function by qualification and depth
- Build a dive site recommendation system based on user preferences